In [1]:
# ============================================================
# Cell 1: Import Required Libraries
# Purpose: Import libraries required for model training
# ============================================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import accuracy_score, f1_score

from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.13.0+cu126
CUDA available: True


In [2]:
# ============================================================
# Cell 2: Set Random Seeds and Device
# Purpose: Ensure reproducibility and select GPU/CPU
# ============================================================

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 3050


In [3]:
# ============================================================
# Cell 3: Load Training and Validation Data
# Purpose: Load the processed datasets created in Notebook 1
# ============================================================

PROJECT_ROOT = "/mnt/g/banglafake-detection"
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

train_path = os.path.join(PROCESSED_DIR, "train.csv")
validation_path = os.path.join(PROCESSED_DIR, "validation.csv")

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

print("Train dataset shape:", train_df.shape)
print("Validation dataset shape:", validation_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts())

print("\nValidation label distribution:")
print(validation_df["label"].value_counts())

Train dataset shape: (2790, 3)
Validation dataset shape: (598, 3)

Train label distribution:
label
1    1397
0    1393
Name: count, dtype: int64

Validation label distribution:
label
1    300
0    298
Name: count, dtype: int64


In [4]:
# ============================================================
# Cell 4: Load BanglaBERT Tokenizer
# Purpose: Convert Bangla text into tokens for BanglaBERT
# ============================================================

from transformers import AutoTokenizer

MODEL_NAME = "csebuetnlp/banglabert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Model:", MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

/home/jaimul/pytorch-env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded successfully!
Model: csebuetnlp/banglabert


In [5]:
# ============================================================
# Cell 5: Test BanglaBERT Tokenization
# Purpose: Verify that Bangla text is correctly tokenized
# ============================================================

sample_text = train_df["text"].iloc[0]

encoding = tokenizer(
    sample_text,
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="pt"
)

print("Original text:")
print(sample_text[:300])

print("\nInput IDs shape:")
print(encoding["input_ids"].shape)

print("\nAttention mask shape:")
print(encoding["attention_mask"].shape)

print("\nNumber of tokens:")
print(int(encoding["attention_mask"].sum()))

Original text:
বড় হচ্ছে ভূ‑পর্যবেক্ষণ প্রযুক্তি বাজার সারা বিশ্বেই ভূ‑পর্যবেক্ষণ প্রযুক্তির চাহিদা বাড়ছে। বড় দেশগুলো তো বটেই উদীয়মান দেশগুলোও এ খাতে তাদের বিনিয়োগ বাড়াচ্ছে। এরই মধ্যে এই প্রযুক্তির বাজার ৫০০ কোটি মার্কিন ডলার ছুঁয়েছে। এটি আগামী ২০৩৩ সালের মধ্যে ৮০০ কোটি ডলার ছাড়িয়ে যাবে বলে জানিয়েছে মহাকাশ প্রযুক্ত

Input IDs shape:
torch.Size([1, 256])

Attention mask shape:
torch.Size([1, 256])

Number of tokens:
256


In [6]:
# ============================================================
# Cell 6: Create Bangla News Dataset Class
# Purpose: Convert Bangla news text into PyTorch tensors
# ============================================================

class BanglaFakeNewsDataset(Dataset):
    
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = str(self.dataframe.loc[idx, "text"])
        label = int(self.dataframe.loc[idx, "label"])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

print("Dataset class created successfully!")

Dataset class created successfully!


In [7]:
# ============================================================
# Cell 7: Create Train & Validation Dataset
# Purpose: Prepare datasets for model training and validation
# ============================================================

MAX_LENGTH = 256

train_dataset = BanglaFakeNewsDataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

validation_dataset = BanglaFakeNewsDataset(
    dataframe=validation_df,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(validation_dataset))

Train dataset size: 2790
Validation dataset size: 598


In [8]:
# ============================================================
# Cell 8: Create DataLoaders
# Purpose: Load training and validation data in mini-batches
# ============================================================

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Batch size:", BATCH_SIZE)

Train batches: 349
Validation batches: 75
Batch size: 8


In [9]:
# ============================================================
# Cell 9: Check Training Batch
# Purpose: Verify DataLoader output shapes and data types
# ============================================================

batch = next(iter(train_loader))

print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["label"].shape)

print("\nInput IDs dtype:", batch["input_ids"].dtype)
print("Attention mask dtype:", batch["attention_mask"].dtype)
print("Labels dtype:", batch["label"].dtype)

print("\nLabels in batch:", batch["label"].tolist())

Input IDs shape: torch.Size([8, 256])
Attention mask shape: torch.Size([8, 256])
Labels shape: torch.Size([8])

Input IDs dtype: torch.int64
Attention mask dtype: torch.int64
Labels dtype: torch.int64

Labels in batch: [1, 0, 0, 0, 1, 0, 0, 1]


In [10]:
# ============================================================
# Cell 10: Load BanglaBERT Backbone
# Purpose: Load pretrained BanglaBERT for contextual embeddings
# ============================================================

from transformers import AutoModel

MODEL_NAME = "csebuetnlp/banglabert"

banglabert = AutoModel.from_pretrained(MODEL_NAME)

banglabert.to(device)

print("BanglaBERT loaded successfully!")
print("Model:", MODEL_NAME)

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

BanglaBERT loaded successfully!
Model: csebuetnlp/banglabert


In [11]:
# ============================================================
# Cell 11: Test BanglaBERT Output
# Purpose: Verify contextual embedding dimensions
# ============================================================

banglabert.eval()

with torch.no_grad():
    sample_input_ids = batch["input_ids"].to(device)
    sample_attention_mask = batch["attention_mask"].to(device)

    bert_output = banglabert(
        input_ids=sample_input_ids,
        attention_mask=sample_attention_mask
    )

print("Last hidden state shape:")
print(bert_output.last_hidden_state.shape)

print("\nBatch size:", bert_output.last_hidden_state.shape[0])
print("Sequence length:", bert_output.last_hidden_state.shape[1])
print("Hidden size:", bert_output.last_hidden_state.shape[2])

Last hidden state shape:
torch.Size([8, 256, 768])

Batch size: 8
Sequence length: 256
Hidden size: 768


In [12]:
# ============================================================
# Cell 12: Define CNN Layer
# Purpose: Extract local patterns from BanglaBERT embeddings
# ============================================================

CNN_CHANNELS = 256
KERNEL_SIZE = 3

cnn = nn.Conv1d(
    in_channels=768,
    out_channels=CNN_CHANNELS,
    kernel_size=KERNEL_SIZE,
    padding=1
).to(device)

print("CNN layer created successfully!")
print("Input channels:", 768)
print("Output channels:", CNN_CHANNELS)
print("Kernel size:", KERNEL_SIZE)

CNN layer created successfully!
Input channels: 768
Output channels: 256
Kernel size: 3


In [14]:
# ============================================================
# Cell 13: Test CNN Output
# Purpose: Pass BanglaBERT embeddings through CNN
# ============================================================

# BanglaBERT output:
# [batch, sequence, hidden] = [8, 256, 768]

bert_embeddings = bert_output.last_hidden_state

# Conv1D expects:
# [batch, channels, sequence]
cnn_input = bert_embeddings.permute(0, 2, 1)

print("Before CNN:", cnn_input.shape)

# Apply CNN
cnn_output = cnn(cnn_input)

# Apply activation
cnn_output = torch.relu(cnn_output)

print("After CNN:", cnn_output.shape)

Before CNN: torch.Size([8, 768, 256])
After CNN: torch.Size([8, 256, 256])


In [15]:
# ============================================================
# Cell 14: Define Attention Layer
# Purpose: Learn which token-level features are important
# ============================================================

ATTENTION_DIM = 128

attention_projection = nn.Linear(
    CNN_CHANNELS,
    ATTENTION_DIM
).to(device)

attention_score = nn.Linear(
    ATTENTION_DIM,
    1
).to(device)

print("Attention layer created successfully!")
print("CNN feature size:", CNN_CHANNELS)
print("Attention dimension:", ATTENTION_DIM)

Attention layer created successfully!
CNN feature size: 256
Attention dimension: 128


In [16]:
# ============================================================
# Cell 15: Calculate Attention Weights
# Purpose: Assign importance weights to each token
# ============================================================

# CNN output shape:
# [batch, channels, sequence]
# [8, 256, 256]

# Convert to:
# [batch, sequence, channels]
attention_input = cnn_output.permute(0, 2, 1)

print("Attention input shape:", attention_input.shape)

# Project CNN features into attention space
attention_hidden = torch.tanh(
    attention_projection(attention_input)
)

print("Attention hidden shape:", attention_hidden.shape)

# Calculate attention scores
attention_scores = attention_score(
    attention_hidden
).squeeze(-1)

print("Attention scores shape:", attention_scores.shape)

# Convert scores into probabilities
attention_weights = torch.softmax(
    attention_scores,
    dim=1
)

print("Attention weights shape:", attention_weights.shape)

# Check that weights sum to 1 for each sample
weight_sum = attention_weights.sum(dim=1)

print("Attention weight sum for first sample:",
      weight_sum[0].item())

Attention input shape: torch.Size([8, 256, 256])
Attention hidden shape: torch.Size([8, 256, 128])
Attention scores shape: torch.Size([8, 256])
Attention weights shape: torch.Size([8, 256])
Attention weight sum for first sample: 1.0


In [17]:
# ============================================================
# Cell 16: Create Attention Context Vector
# Purpose: Combine token features using learned attention weights
# ============================================================

# Expand attention weights:
# [batch, sequence] → [batch, sequence, 1]

attention_weights_expanded = attention_weights.unsqueeze(-1)

# Weighted CNN features
weighted_features = (
    attention_input * attention_weights_expanded
)

print("Weighted features shape:", weighted_features.shape)

# Sum across the sequence dimension
attention_context = weighted_features.sum(dim=1)

print("Attention context shape:", attention_context.shape)

Weighted features shape: torch.Size([8, 256, 256])
Attention context shape: torch.Size([8, 256])
